# LLM Response Preference Classification

## Results Summary

| Model | Val loss | Val acc | Test loss | Test acc| Kaggle score|
|-|-|-|-|-|-|
|Cross-encoder, no training|1.3955996409006723|34.20320111343076|-|-|-|
|Cross-encoder, head only|1.0757560135178377|40.04871259568545|-|-|-|
|Cross-encoder, full fine-tuned |1.0324580619628843|47.14683368128045|1.0233467391736821|47.529575504523315|-|

Cross-encoder, head only reached peak at epoch 91 out of 160 epochs
Cross-encoder, full fine-tuning reached peak at epoch 3 out of 20

## Setup: imports, configuration, and experiment tracking

In [1]:
!pip install -q mlflow
!pip install -q ipywidgets


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import gc
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import random
import warnings

try:
    import mlflow
    MLFLOW_AVAILABLE = True
except ImportError:
    MLFLOW_AVAILABLE = False
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from functools import partial
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

from models import DeepPreferenceClassifier, PreferenceModel
from preference_datasets import (
    CrossEncoderDataset,
    collate_cross_encoder,
    EncodedPreferenceDataset
)
from text_utils import (
    parse_conversation,
    format_conversation,
    token_lengths,
    summarize
)
from training import (
    train_model,
    validate_epoch,
    train_classifier_model,
    set_seed
)

from plotting import plot_results

SEED = 42
generator = set_seed(SEED)

# Environment
ENVIRONMENT = "MAC" # "COLAB", "KAGGLE", "MAC"
if ENVIRONMENT == "KAGGLE":
    TARGET_DEVICE = "cuda"
    PROJECT_ROOT = "./"
    TOKENIZER_PRETRAINED = "/kaggle/input/datasets/fernandoquintinojr/basic-models-deberta/deberta-v3-small-tokenizer"
    ENCODER_PRETRAINED = "/kaggle/input/datasets/fernandoquintinojr/basic-models-deberta/deberta-v3-small-encoder"
elif ENVIRONMENT == "MAC":
    TARGET_DEVICE = "mps"
    PROJECT_ROOT = "./"
    TOKENIZER_PRETRAINED = "microsoft/deberta-v3-small"
    ENCODER_PRETRAINED = "microsoft/deberta-v3-small"
elif ENVIRONMENT == "COLAB":
    TARGET_DEVICE = "cuda"
    PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/LLM Classification Finetuning"
    from google.colab import drive
    drive.mount('/content/drive')
    TOKENIZER_PRETRAINED = "microsoft/deberta-v3-small"
    ENCODER_PRETRAINED = "microsoft/deberta-v3-small"


# Training
if ENVIRONMENT == "KAGGLE":
    BATCH_SIZE = 128
    NUM_WORKERS = 2
    MULTIPROCESSING_CONTEXT = None
elif ENVIRONMENT == "MAC":
    # To be used only when training the classifier with pre-encoded input.
    BATCH_SIZE = 512
    NUM_WORKERS = 6
    MULTIPROCESSING_CONTEXT = "fork"
elif ENVIRONMENT == "COLAB":
    BATCH_SIZE = 128
    NUM_WORKERS = 0
    MULTIPROCESSING_CONTEXT = None


device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device}")

if device.type != TARGET_DEVICE:
    print(f"You are currently using device: {device}. If you will be training, please use {TARGET_DEVICE}")

Using mps


In [ ]:
# CLASSIFIER_WEIGHTS_FILE must be a full/absolute path. A relative path would
# resolve differently depending on where the process is run from.
if ENVIRONMENT == "KAGGLE":
    DATA_PATH = "/kaggle/input/competitions/llm-classification-finetuning" # Folder containing data
    CLASSIFIER_WEIGHTS_FILE = (
        "/kaggle/input/models/fernandoquintinojr/classifier-model-llm/pytorch/default/1/classifier_state_1_state_dict.pth"
    )
    CROSS_FULL_MODEL_WEIGHTS_FILE = "/kaggle/input/models/fernandoquintinojr/cross-encoder-llm-response-classifier/pytorch/default/1/cross_encoder_full_model_trained.pt"
elif ENVIRONMENT == "MAC":
    DATA_PATH = "data" # Folder containing data
    CLASSIFIER_WEIGHTS_FILE = "trained_cross_encoder_models/model_classifier-state-dict.pt"
    CROSS_FULL_MODEL_WEIGHTS_FILE = "trained_cross_encoder_models/cross_encoder_full_model_trained.pt"
elif ENVIRONMENT == "COLAB":
    DATA_PATH = "data" # Folder containing data
    CLASSIFIER_WEIGHTS_FILE = os.path.join(
        PROJECT_ROOT,
        "trained_cross_encoder_models/model_classifier-state-dict.pt"
    )
    CROSS_FULL_MODEL_WEIGHTS_FILE = "trained_cross_encoder_models/cross_encoder_full_model_trained.pt"

print(f"ENVIRONMENT: {ENVIRONMENT}")
print("Data location:", os.path.join(PROJECT_ROOT,DATA_PATH))
print("DATA_PATH exists:", os.path.exists(os.path.join(PROJECT_ROOT,DATA_PATH)))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", os.path.exists(PROJECT_ROOT))
print("CLASSIFIER_WEIGHTS_FILE: ", CLASSIFIER_WEIGHTS_FILE)
print(
    "CLASSIFIER_WEIGHTS_FILE exists:",
    os.path.exists(CLASSIFIER_WEIGHTS_FILE)
    )
print(
    "CROSS_FULL_MODEL_WEIGHTS_FILE exists:",
    os.path.exists(CROSS_FULL_MODEL_WEIGHTS_FILE)
    )


ENVIRONMENT: MAC
Data location: ./data
DATA_PATH exists: True
PROJECT_ROOT: ./
PROJECT_ROOT exists: True
CLASSIFIER_WEIGHTS_FILE:  trained_cross_encoder_models/model_classifier-state-dict.pt
CLASSIFIER_WEIGHTS_FILE exists: True
CROSS_FULL_MODEL_WEIGHTS_FILE exists: True


In [4]:
if MLFLOW_AVAILABLE:
    # SQLite backend (vs. MLflow's older file-store) - must be a genuine local
    # path, not cloud-synced storage (e.g. Google Drive), which caused disk I/O
    # errors due to SQLite's locking requirements.
    mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT}/mlruns_v2")
    EXPERIMENT_NAME = f"preference-classifier-{ENVIRONMENT}"
    if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
        mlflow.create_experiment(EXPERIMENT_NAME,
                                artifact_location=f"{PROJECT_ROOT}/mlartifacts")
    mlflow.set_experiment(EXPERIMENT_NAME)

## The Data

In [5]:
df = pd.read_csv(os.path.join(PROJECT_ROOT, DATA_PATH, "train.csv"))
DATA_CLEAN = False
# Convert one-hot winner columns to a single label via argmax:
# 0=response_a, 1=response_b, 2=tie.
labels_one_hot = df[["winner_model_a", "winner_model_b", "winner_tie"]]
labels = labels_one_hot.values.argmax(axis=1)

train_df, val_df, train_labels, val_labels = train_test_split(
    df,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=SEED
)
val_df, test_df, val_labels, test_labels = train_test_split(
    val_df,
    val_labels,
    test_size=0.5,
    stratify=val_labels,
    random_state=SEED
)


In [6]:
idx = 20
if not DATA_CLEAN:
    print("Text before cleaning:")

    print(train_df["prompt"].iloc[idx])
    print("\nCleaning Data")
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=SyntaxWarning)
        for col in ["prompt",  "response_a", "response_b"]:
            train_df[col] = train_df[col].apply(parse_conversation).apply(format_conversation)
            val_df[col] = val_df[col].apply(parse_conversation).apply(format_conversation)
            test_df[col] = test_df[col].apply(parse_conversation).apply(format_conversation)
            df[col] = df[col].apply(parse_conversation).apply(format_conversation)
    DATA_CLEAN = True

print("\nText after cleaning:")
print(train_df["prompt"].iloc[idx])

Text before cleaning:
["Name some major events of the 20th century that did not happen in Europe and America","Tell me more about the Chinese Communist Revolution"]

Cleaning Data

Text after cleaning:
Name some major events of the 20th century that did not happen in Europe and America

Tell me more about the Chinese Communist Revolution


## Model Architecture


**Input variables**
```
prompt = prompt
response_0 = either response_a or response_b
response_1 = the other response_a/b
input = [CLS] prompt [SEP] response_0 [SEP] response_1 [SEP]
```
**Models**
```
model_encoder = DeBERTA v3 small
model_classifier = DeepPreferenceClassifier 3 blocks
model = PreferenceModel (wraps the encoder and classifier)
```
**Pipeline**
```
input -> model -> logits
```
The PreferenceModel class wraps the encoder and classifier as follows:
```
input -> model_encoder -> encoded

encoded -> model_classifier -> logits
```



In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PRETRAINED)

prompt_lens = token_lengths(train_df["prompt"].tolist(), tokenizer)
response_a_lens = token_lengths(train_df["response_a"].tolist(), tokenizer)
response_b_lens = token_lengths(train_df["response_b"].tolist(), tokenizer)

summarize("prompt", prompt_lens)
summarize("response_a", response_a_lens)
summarize("response_b", response_b_lens)


prompt: mean=80.1, median=20, 95th=316, 99th=1046, max=6584
response_a: mean=289.4, median=227, 95th=798, 99th=1470, max=6377
response_b: mean=291.4, median=229, 95th=790, 99th=1518, max=15670


### Datasets

### Models

## Training

In [ ]:
model_encoder = AutoModel.from_pretrained(ENCODER_PRETRAINED, torch_dtype=torch.float32)
model_classifier = DeepPreferenceClassifier(input_dim=768, num_blocks=3)
model = PreferenceModel(model_encoder,model_classifier)
model = model.to(device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
run_validation = False

if run_validation:
    print("--Validation before training--")
    loss = nn.CrossEntropyLoss()

    val_dataset = CrossEncoderDataset(
        prompts=val_df["prompt"].tolist(),
        responses_a=val_df["response_a"].tolist(),
        responses_b=val_df["response_b"].tolist(),
        labels=val_labels,
        tokenizer=tokenizer,
        prob=0,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        multiprocessing_context=MULTIPROCESSING_CONTEXT,
        collate_fn=partial(collate_cross_encoder, tokenizer=tokenizer),
        generator=generator,
    )


    # Validation before training
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()
    if device.type == TARGET_DEVICE:
        model = model.to(device)
        epoch_loss, epoch_acc = validate_epoch(model, val_loader, device, loss)
        print(f"Validation loss: {epoch_loss}")
        print(f"Validation acc: {epoch_acc}")
    else:
        print(
            f"Device is set to {device}. Needs to be set to {TARGET_DEVICE}."
        )

### Stage 0: Build the embedding cache

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PRETRAINED)

full_dataset = CrossEncoderDataset(
    prompts=df["prompt"].tolist(),
    responses_a=df["response_a"].tolist(),
    responses_b=df["response_b"].tolist(),
    labels=labels,
    tokenizer=tokenizer,
    prob=0
)

full_loader = DataLoader(
    full_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=partial(collate_cross_encoder, tokenizer=tokenizer),
    num_workers=6,
    multiprocessing_context=MULTIPROCESSING_CONTEXT,
)

In [ ]:
# Preprocess input through encoder:
run_encoding = False # Warning: Will replace existing file
encoded_filename = os.path.join(PROJECT_ROOT, "cross-encoded_data.pt")
print(f"Encoded file: {encoded_filename}")
print(f"File exists: {os.path.exists(encoded_filename)}")
if run_encoding:
    print("\n--Encoding the input for fast classifier training--")
    model_encoder = AutoModel.from_pretrained(
        ENCODER_PRETRAINED,
        torch_dtype=torch.float32,
    )

    model_encoder = model_encoder.to(device)
    print(f"Encoding using device: {device}")
    model_encoder.eval()

    hidden_size = model_encoder.config.hidden_size
    num_samples = len(full_dataset)
    encoded = torch.zeros(num_samples, hidden_size)
    dataset_idx =torch.zeros(num_samples, dtype=torch.long)

    with torch.no_grad():
        for batch in tqdm(full_loader):
            idx = batch["idx"]
            batch = {k: v.to(device) for k,v in batch.items() if k != "idx"}
            with torch.autocast(device_type=device.type, dtype=torch.float16):
                output = model_encoder(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"]
                ).last_hidden_state[:,0,:]
            encoded[idx] = output.float().cpu()
            dataset_idx[idx] =idx.cpu()

    print("Saving...")
    torch.save(
        {"encoded": encoded, "label": torch.tensor(labels), "idx": dataset_idx},
        encoded_filename
    )

    print("Saved file")

Encoded file: ./cross-encoded_data.pt
File exists: True


### Stage 1: Pretrain the classifier head

<img src="images/loss_history_classifier_cross.png">
<img src="images/acc_history_classifier_cross.png">




In [ ]:
if os.path.exists(encoded_filename):
    print("---Caching preencoded data---")
    cache = torch.load(encoded_filename)
    encoded, labels_cached = cache["encoded"], cache["label"]

    train_encoded_dataset = EncodedPreferenceDataset(
        encoded=encoded,
        labels=labels_cached,
        indices=train_df.index.values
    )
    val_encoded_dataset = EncodedPreferenceDataset(
        encoded=encoded,
        labels=labels_cached,
        indices=val_df.index.values,
    )
else:
    print("Encoded file does not Exist.")


In [ ]:
blocks = [3,]
run_training = False
results_history =[]

if run_training:
    for block in blocks:
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        elif device.type == "mps":
            torch.mps.empty_cache()
        run_name = f"Cross-Encoder-classifier-{block}-blocks"

        print(run_name)


        # Reset loaders
        generator = set_seed(SEED)
        train_encoded_loader = DataLoader(
            train_encoded_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            generator=generator,
        )
        val_encoded_loader = DataLoader(
            val_encoded_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            generator=generator,
        )

        # Reset model
        model = DeepPreferenceClassifier(input_dim=768, num_blocks=block)
        scaler = torch.amp.GradScaler(device.type)
        loss = nn.CrossEntropyLoss()
        learning_rate = 1e-3
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",  # Decrease val loss.
            factor=0.5,
            patience=3,
        )
        # Confirm the runtime actually has the intended device and has a
        # run name before running training.
        if device.type == TARGET_DEVICE and run_name is not None:
            results = train_classifier_model(model=model,
                                train_loader=train_encoded_loader,
                                val_loader=val_encoded_loader,
                                device=device,
                                optimizer=optimizer,
                                loss=loss,
                                num_epochs=160,
                                scaler=scaler,
                                scheduler=scheduler,
                                run_name=run_name
            )
            results_history.append(results)
            if len(blocks) == 1:
                CLASSIFIER_WEIGHTS_FILE = os.path.join(
                    PROJECT_ROOT,
                    f"model_classifier-state-dict.pt"
                )
                torch.save(
                results["best_model"].state_dict(),
                CLASSIFIER_WEIGHTS_FILE
                )
                print(f"\nSaved model as: {CLASSIFIER_WEIGHTS_FILE}\n")
                plot_results(results)
        else:
            print(
                f"Device is set to {device} (Needs to be set to {TARGET_DEVICE})."
                )

### Stage 2: Full fine-tuning

<img src="images/acc_history_full_model_cross.png">
<img src="images/loss_history_full_model_cross.png">


ADD TO README: Run Stage 1, then Stage 2, in order, to reproduce the reported results.

In [47]:
model_encoder = AutoModel.from_pretrained(ENCODER_PRETRAINED, torch_dtype=torch.float32)
model_encoder.gradient_checkpointing_enable()
model_classifier = DeepPreferenceClassifier(input_dim=768, num_blocks=3)
model_classifier.load_state_dict(torch.load(CLASSIFIER_WEIGHTS_FILE, map_location="cpu"))
model = PreferenceModel(model_encoder,model_classifier)


NameError: name 'ENCODER_PRETRAINED' is not defined

In [59]:
train_dataset = CrossEncoderDataset(
    prompts=train_df["prompt"].tolist(),
    responses_a=train_df["response_a"].tolist(),
    responses_b=train_df["response_b"].tolist(),
    labels=train_labels,
    tokenizer=tokenizer,
)
val_dataset = CrossEncoderDataset(
    prompts=val_df["prompt"].tolist(),
    responses_a=val_df["response_a"].tolist(),
    responses_b=val_df["response_b"].tolist(),
    labels=val_labels,
    tokenizer=tokenizer,
    prob=0,
)
test_dataset = CrossEncoderDataset(
    prompts=test_df["prompt"].tolist(),
    responses_a=test_df["response_a"].tolist(),
    responses_b=test_df["response_b"].tolist(),
    labels=test_labels,
    tokenizer=tokenizer,
    prob=0,
)

In [49]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    multiprocessing_context=MULTIPROCESSING_CONTEXT,
    collate_fn=partial(collate_cross_encoder, tokenizer=tokenizer),
    generator=generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    multiprocessing_context=MULTIPROCESSING_CONTEXT,
    collate_fn=partial(collate_cross_encoder, tokenizer=tokenizer),
    generator=generator,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    multiprocessing_context=MULTIPROCESSING_CONTEXT,
    collate_fn=partial(collate_cross_encoder, tokenizer=tokenizer),
    generator=generator,
)

In [10]:
run_validation = False
if run_validation:
    print("--Validation on trained classifier--")
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()

    loss = nn.CrossEntropyLoss()
    if device.type == TARGET_DEVICE:
        epoch_loss, epoch_acc = validate_epoch(
            model=model,
            loader=val_loader,
            device=device,
            loss=loss,
        )
        print(f"Validation loss: {epoch_loss}")
        print(f"Validation acc: {epoch_acc}")
    else:
        print(f"Device {device} does not match target device{TARGET_DEVICE}.")


In [ ]:
run_name = f"full-finetuning"
run_training = False
if run_training:
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()


    print(run_name)
    # Reset randomness
    generator = set_seed(SEED)

    scaler = torch.amp.GradScaler(device.type)
    loss = nn.CrossEntropyLoss()
    learning_rate = 2e-5
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",  # Decrease val loss.
        factor=0.5,
        patience=3,
    )
    # Confirm the runtime actually has the intended device and has a run name
    # before running training.
    if device.type == TARGET_DEVICE and run_name is not None:
        results = train_model(model=model,
                            train_loader=train_loader,
                            val_loader=val_loader,
                            device=device,
                            optimizer=optimizer,
                            loss=loss,
                            num_epochs=5,
                            scaler=scaler,
                            scheduler=scheduler,
                            run_name=run_name
        )

        FULL_WEIGHTS_FILE = os.path.join(PROJECT_ROOT, f"full-model-loss-{results["best_loss"]:.4f}.pt")
        torch.save(
        results["best_model"].state_dict(),
        FULL_WEIGHTS_FILE
        )
        print(f"\nSaved model as: {FULL_WEIGHTS_FILE}\n")
        plot_results(results)
    else:
        print(f"Device is set to {device} (Needs to be set to {TARGET_DEVICE}).")

In [ ]:
model_encoder = AutoModel.from_pretrained(ENCODER_PRETRAINED, torch_dtype=torch.float32)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PRETRAINED)

tokenizer.save_pretrained("./deberta-v3-small-tokenizer")
model_encoder.save_pretrained("./deberta-v3-small-encoder")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
model_encoder = AutoModel.from_pretrained(ENCODER_PRETRAINED, torch_dtype=torch.float32)
model_encoder.gradient_checkpointing_enable()
model_classifier = DeepPreferenceClassifier(input_dim=768, num_blocks=3)
model = PreferenceModel(model_encoder,model_classifier)
model.load_state_dict(torch.load(CROSS_FULL_MODEL_WEIGHTS_FILE, map_location="cpu"))
model = model.to(device)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<All keys matched successfully>

In [11]:
run_validation = False
if run_validation:
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()

    loss = nn.CrossEntropyLoss()
    if device.type == TARGET_DEVICE:
        epoch_loss, epoch_acc = validate_epoch(
            model=model,
            loader=test_loader,
            device=device,
            loss=loss,
        )
        print(f"Test loss: {epoch_loss}")
        print(f"Test acc: {epoch_acc}")
    else:
        print(f"Device {device} does not match target device{TARGET_DEVICE}.")


In [2]:
run_submission = False
if run_submission:
    submission_df = pd.read_csv(os.path.join(PROJECT_ROOT, DATA_PATH, "test.csv"))
    SUBMISSION_DATA_CLEAN = False


In [3]:
if run_submission:
    if not SUBMISSION_DATA_CLEAN:
        print("Text before cleaning:")

        print("\nCleaning Data")
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=SyntaxWarning)
            for col in ["prompt",  "response_a", "response_b"]:
                submission_df[col] = submission_df[col].apply(parse_conversation).apply(format_conversation)
        SUBMISSION_DATA_CLEAN = True
        print("Text is now clean.")

In [ ]:
if run_submission:
    dummy_labels = np.zeros(len(submission_df), dtype=int)

    submission_dataset = CrossEncoderDataset(
        prompts=submission_df["prompt"].tolist(),
        responses_a=submission_df["response_a"].tolist(),
        responses_b=submission_df["response_b"].tolist(),
        labels=dummy_labels,
        tokenizer=tokenizer,
        prob=0
    )

    submission_loader = DataLoader(
        submission_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        multiprocessing_context=MULTIPROCESSING_CONTEXT,
        collate_fn=partial(collate_cross_encoder, tokenizer=tokenizer),
    )

In [ ]:


if run_submission:
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()
    prediction_prob =[]
    prediction_idx = []
    if device.type == TARGET_DEVICE:
        model.eval()
        with torch.no_grad():
                for batch in tqdm(submission_loader):
                    batch = {k: v.to(device) for k, v in batch.items()}
                    output = model(
                        batch["input_ids"],
                        batch["attention_mask"]
                    )
                    probs = torch.softmax(output, dim=1)

                    prediction_prob.append(probs)
                    prediction_idx.append(batch["idx"])
        all_probs = torch.cat(prediction_prob, dim=0)
        all_probs = all_probs.cpu().numpy()
        all_idx = torch.cat(prediction_idx, dim=0)
        all_idx = all_idx.cpu().numpy()
        submission_ids = submission_df["id"].values[all_idx]

        submission = pd.DataFrame({
              "id": submission_ids,
              "winner_model_a": all_probs[:,0],
              "winner_model_b": all_probs[:,1],
              "winner_tie": all_probs[:,2],
        })

        submission.to_csv("submission.csv", index=False)
    else:
        print(f"Device {device} does not match target device{TARGET_DEVICE}.")


In [33]:
#runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
#runs